In [2]:
from langgraph.graph import StateGraph, START, END
# from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    # repo_id="LiquidAI/LFM2.5-230M",
    task="text-generation",
)
model = ChatHuggingFace(llm=llm)

In [5]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [6]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [7]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [8]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The average distance from the Earth to the Moon is about 384,400 kilometers (238,855 miles). However, the actual distance can vary due to the elliptical orbit of the Moon around the Earth. At its closest point (perigee), the Moon is about 363,104 kilometers (225,623 miles) from Earth, and at its farthest point (apogee), it is about 405,696 kilometers (252,088 miles) away.


In [9]:
model.invoke('How far is moon from the earth?').content

"The average distance from the Moon to the Earth is approximately 384,400 kilometers (238,855 miles). However, this distance can vary slightly due to the elliptical nature of the Moon's orbit around the Earth. At its closest point (perigee), the Moon can be as close as about 363,104 kilometers (225,623 miles) from Earth, and at its farthest point (apogee), it can be about 405,696 kilometers (252,088 miles) away."